# 空间变换模型 Spatial Transformer Networks

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/logo.png" width=150>

空间变换网络（STN）允许神经网络学习对输入图像进行空间变换（如旋转、缩放、仿射变换），使模型能够主动处理几何变换。

Spatial Transformer Networks (STN) allow neural networks to learn spatial transformations (such as rotation, scaling, affine transformations) on input images, enabling the model to actively handle geometric variations.

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/stn.png" width=500>

# 概述 Overview

* **目标:**  让神经网络学习对输入进行空间变换以改善分类。
* **优点:** 
  * 可端到端训练
  * 适用于各种几何变换
  * 增强模型鲁棒性
* **缺点:**
  * 增加计算开销
  * 需要额外的定位网络
* **其他:** 
  * 是注意力机制的早期形式
  * 广泛应用于图像识别

# 设置 Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# 空间变换器 Spatial Transformer

In [ ]:
class SpatialTransformer(nn.Module):
    def __init__(self, input_dim):
        super(SpatialTransformer, self).__init__()
        
        # 定位网络 Localization network
        self.localization = nn.Sequential(
            nn.Conv2d(input_dim, 32, kernel_size=3, padding=1),
            nn.MaxPool2d(2, 2),
            nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.MaxPool2d(2, 2),
            nn.ReLU(True)
        )
        
        # 回归网络 regressor for transformation parameters
        self.fc_loc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(True),
            nn.Linear(128, 6)  # 6 个仿射变换参数 6 affine transformation parameters
        )
        
        # 初始化为单位矩阵 Initialize as identity transformation
        self.fc_loc[2].weight.data.fill_(0)
        self.fc_loc[2].bias.data = torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float)
    
    def forward(self, x):
        # 获取变换参数 Get transformation parameters
        xs = self.localization(x)
        xs = xs.view(-1, 64 * 7 * 7)
        theta = self.fc_loc(xs)
        theta = theta.view(-1, 2, 3)
        
        # 应用仿射变换 Apply affine transformation
        grid = F.affine_grid(theta, x.size(), align_corners=True)
        x_transformed = F.grid_sample(x, grid, align_corners=True)
        
        return x_transformed

# 带 STN 的 CNN CNN with STN

In [ ]:
class CNNwithSTN(nn.Module):
    def __init__(self, num_classes=10):
        super(CNNwithSTN, self).__init__()
        
        # 空间变换器 Spatial Transformer
        self.stn = SpatialTransformer(1)  # 输入通道数 input channels
        
        # 卷积层 Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        
        # 全连接层 FC layers
        self.fc1 = nn.Linear(64 * 5 * 5, 128)
        self.fc2 = nn.Linear(128, num_classes)
    
    def forward(self, x):
        # 应用 STN Apply STN
        x = self.stn(x)
        
        # 标准 CNN 流程 Standard CNN process
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = x.view(-1, 64 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

# 训练示例 Training Example

In [ ]:
# 加载 MNIST 数据 Load MNIST data
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

# 初始化 Initialize
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CNNwithSTN(num_classes=10).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)


In [ ]:
# 训练循环 Training loop
for epoch in range(5):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
    
    print(f'Epoch [{epoch+1}/5], Loss: {loss.item():.4f}')

# TODO

- 可变形卷积 Deformable Convolutions
- 空间注意力 Spatial Attention
- STN 在目标检测中的应用 STN in Object Detection